In [1]:
import os
import struct
import argparse

TAG_FORMAT = '3s30s30s30s4s28sBBB'


def process_mp3(directory, show_dump=False, set_genre=None):
    files = [f for f in os.listdir(directory) if f.endswith('.mp3')]

    if not files:
        print("MP3 файлы не найдены.")
        return

    track_counter = 1

    for filename in files:
        filepath = os.path.join(directory, filename)

        with open(filepath, 'rb+') as f:
            f.seek(-128, 2)
            tag_data = f.read(128)

            fields = struct.unpack(TAG_FORMAT, tag_data)

            header = fields[0].decode('ascii', errors='ignore')

            if header != 'TAG':
                print(f"Файл {filename}: ID3v1 тег не найден.")
                continue

            title = fields[1].strip(b'\x00').decode('cp1251', errors='ignore')
            artist = fields[2].strip(b'\x00').decode('cp1251', errors='ignore')
            album = fields[3].strip(b'\x00').decode('cp1251', errors='ignore')
            year = fields[4].strip(b'\x00').decode('cp1251', errors='ignore')
            comment = fields[5].strip(b'\x00').decode('cp1251', errors='ignore')
            zero_byte = fields[6]
            track = fields[7]
            genre = fields[8]

            print(f"[{artist}] - [{title}] - [{album}]")

            if show_dump:
                print(f"  HEX DUMP: {tag_data.hex(' ')}")

            updated = False
            new_track = track
            new_genre = genre

            if track == 0:
                new_track = track_counter
                updated = True

            if genre == 255 and set_genre is not None:
                new_genre = set_genre
                updated = True

            if updated:
                new_tag = struct.pack(TAG_FORMAT,
                                      fields[0], fields[1], fields[2], fields[3],
                                      fields[4], fields[5], 0, new_track, new_genre
                                      )
                f.seek(-128, 2)
                f.write(new_tag)
                print(f"  -> Теги обновлены: Трек {new_track}, Жанр {new_genre}")

            track_counter += 1


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("dir", help="Директория с MP3 файлами")
    parser.add_argument("-d", "--dump", action="store_true", help="Показать HEX дамп")
    parser.add_argument("-g", "--genre", type=int, help="Установить номер жанра")

    args = parser.parse_args()
    process_mp3(args.dir, args.dump, args.genre)

usage: ipykernel_launcher.py [-h] [-d] [-g GENRE] dir
ipykernel_launcher.py: error: unrecognized arguments: -f


SystemExit: 2

D:\Computer\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
